# Úloha č.1:

### Znenie

Implementuje overenie toho, či vaše zariadenie disponuje CUDA-enabled GPU. Ak áno, použite ho pri implementácii siete a jej trénovaní/testovaní.

### Riešenie

In [ ]:
 device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Vysvetlenie

Bol vytvorený objekt *troch.device*, slúžiaci na špecifikovanie zariadenia, na ktorom budú dáta/model siete ukladané. Objekt dostáva ako svoj vstupný parameter názov požadovaného zariadnia, za pomoci ternárneho operátora a funkcie *torch.cuda.is_available()*, ktorá vracia *True*, ak vaše zariadenie obsahuje CUDA-enabled GPU.

***

# Úloha č.2

### Znenie

Implementujte funkciu *clean_text(text)*, ktorá vstupný text prevedie na malé písmená a odstráni z neho všetky znaky okrem alfanumerických znakov, nasledujúcich znakov: *?* *!* a medzier. Môžete pri tom použiť napríklad regulárne výrazy.

### Riešenie

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z0-9\?!\'\\s]+','', text)
    
    return text

print(clean_text("Mac.ka 123!"))

### Vysvetlenie

Vytvorená funkcia na čistenie textu je pomerne jednoduchá. Prevedenie textu na malé písmená vykonáva pomocou vstavanej funkcie Pythonu *.lower()*, zbavenie textu nežiadúcich znakov pomocou regulárneho výrazu, za pomoci knižnice *re*. Prácu s regulárnym výrazom, ako aj jeho presnejší opis môžete vidieť na tomto [linku](https://regex101.com/r/cjE11g/1). 

***

# Úloha č.3:

### Znenie

Načítajte text zo súboru *jokes.txt*, iterujte ním po riadkoch pričom každý riadok vyčistite implementovanou funkciou a vyčistený text rozdeľte na tokeny a získaný zoznam uložte ich do poľa *all_jokes*.

> ##### ***Poznámka***
*Pozn: do poľa ukladajte tokeny jednotlivých vtipov vo vnorených poliach, nie len ich elementy*

### Riešenie

In [ ]:
all_jokes = []
tokenizer = get_tokenizer("basic_english")

with open("./Data/Jokes/jokes.txt", 'r') as f:
    for line in f:
        clear_line = clean_text(line)
        line_tokens = tokenizer(clear_line)
        
        all_jokes.append(line_tokens)

### Vysvetlenie

Vytvorený cyklus jednoducho iteruje riadok po riadku text obsiahnutý v súbore *jokes.txt*. Každý z vtipov následne vyčistí, pomocou tokenizera rozdelí na tokeny a výsledný zoznam pridá do poľa všetkých vtipov. Všimnite si, že bola použitá funkcia *append* a nie jednoduché sčítavanie polí. To má za následok to, že pole *all_jokes* bude poľom polí obsahujúcich tokeny, tak ako to to stojí v znení úlohy.

***

# Úloha č.4:

### Znenie

Kód predchádzajúcej rozšírte o vytvorenie counter-u, ktorý iteratívne rozširujte o tokeny jednotlivých vtipov. Po načítaní všetkých vtipov získajte zoznam unikátnych tokenov všetkých vtipov do poľa unique_tokens. Ten bude predstavovať vašu slovnú zásobu.

### Riešenie

In [ ]:
all_jokes = []
counter = Counter()

with open("./Data/Jokes/jokes.txt", 'r') as f:
    for line in f:
        clear_line = clean_text(line)
        line_tokens = tokenizer(clear_line)
        
        all_jokes.append(line_tokens)
        counter.update(line_tokens)
        
unique_tokens = list(counter)

### Vysvetlenie

Do kódu z predchádzajúcej úlohy bola pridaná implementácia objektu *Counter*, ktorý je iteratívne rozširovaný o tokeny vtipov. Slovná zásoba je z neho vytvorená po ukončení cyklu prevedením na list. Ide o jeden z možných spôsobov, ako získať zoznam unikátnych tokenov z celého spracovávaného textu.

***

# Úloha č.5:

### Znenie

Kód z predchádzajúcej úlohy rozšírte o vytvorenie nasledujúcich dvoch slovníkov:
1. **token_to_index str->int** - mapuje tokeny na čísla
2. **index_to_token int->str** - mapuje čísla na tokeny

Môže vám pri tom poslúžiť napríklad metóda *enumerate()*

Po vytvorení týchto slovníkov namapujte dáta v poli *all_jokes* na ich číselné reprezentácie. Nezabudnite však udržať pôvodnú 2D štruktúru poľa.

### Riešenie

In [ ]:
from collections import Counter

all_jokes = []
counter = Counter()
tokenizer = get_tokenizer("basic_english")

with open("./Data/Jokes/jokes.txt", 'r') as f:
    for line in f:
        clear_line = clean_text(line)
        line_tokens = tokenizer(clear_line)
        
        all_jokes.append(line_tokens)
        counter.update(line_tokens)
        
unique_tokens = list(counter)

index_to_token = {index: word for index, word in enumerate(unique_tokens)}
token_to_index = {word: index for index, word in enumerate(unique_tokens)}

all_jokes = [
    [
        token_to_index[token] for token in joke
    ] for joke in all_jokes
]

### Vysvetlenie

Takto rozšírený kód obsahuje vytvorenie dvoch slovníkov, tak ako boli opísané v znení úlohy. Vytvorené slovníky umožňujú vykonávať konverziu stringových tokenov na ich prislúchajúci celočíselný index a naopak. Konverzia z tokenu na index je následne využitá na prekonvertovanie poľa all_jokes, z poľa polí tokenov na pole polí indexov. Takáto reprezentácia umožňuje s dátami ďalej pracovať pri vytváraní Datasetu.

***

# Úloha č.6:

### Znenie

Implementuje triedu *JokesDataset*, ktorá spracuje dáta z poľa *all_jokes* podľa hore ukázaného princípu, s vami určenou dĺžkou sekvencie slov. Následne vytvorte pomocou nej objekt *Dataloader*.

### Riešenie

In [ ]:
# implement dataset
class JokesDataset(Dataset):
    def __init__(self, sequence_length, jokes):
        self.sequence_length = sequence_length        
        self.data = []
        self.labels = []
        
        for joke in jokes:
            for i in range(len(joke) - (self.sequence_length)):
                features = joke[i: i + self.sequence_length]
                label = joke[i + self.sequence_length]

                self.data.append(torch.Tensor(features).int())
                self.labels.append(label) 
    
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, index):
        return self.data[index], self.labels[index]
        
# define hyperparameters
batch_size = 100 
seq_len = 4

# initialize dataloader
dataset = JokesDataset(seq_len, all_jokes)

data_loader = DataLoader(
    dataset = dataset,
    batch_size = batch_size,
    shuffle = True
)

### Vysvetlenie

Uvedený kód predstavuje tvorbu triedy rozširujúcej triedu *Dataset*, obsahujúcej vlastné dáta budované zo spracovaného textu. 

V konštruktore triedy sú inicializované dve polia, ktoré sú následne iteratívne napĺňané dátami, ktoré vyplývajú z už vytvoreného zoznamu vtipov *all_jokes*. Z každého z vtipov sú postupne vytvárané tensory sekvencií zvolenej dĺžky, ktoré sú pridávané do poľa *data*. Za triedu sekvencie je považované slovo, ktoré sa vo vete nachádza ako nasledujúce za sekvenciou. Takáto implementácia je v súlade s princípom budovania datasetu, tak ako je opísaný na cvičení.

***

# Úloha č.7:

### Znenie

Implementuje trénovaciu slučke pre vami vytovernú sieť. Stratovú funkciu, optimizátor ako aj hyperparametre trénovania si zvoľte sami a sieť trénujte na dátach z vami iplmentovaného JokesDaset setu. Dbajte na vhodnú veľkosť tensorov vstupujúcu do siete.

### Riešenie

In [ ]:
model=lstm(vocab_size=len(unique_tokens),embed_size=128, hidden_size=256)

num_epochs = 20
learning_rate = 0.005

criterion = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) 

start_time = time.time()
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1} training progress:")

    for i, (sequences, labels) in enumerate(tqdm(data_loader)): 
        reshaped = sequences.reshape(-1, seq_len).to(device)
        labels = labels.to(device)
                
        outputs = model(sequences) 
        optimizer.zero_grad() 
                
        loss = criterion(outputs, labels) 
        loss.backward() 
        
        optimizer.step() 
    
    if epoch + 1 == num_epochs: 
        print (f'\nTraining took {str(timedelta(seconds=time.time() - start_time))}, loss after all epochs: {loss.item():.5f}\n')    
    else:
        print (f'Loss at end of epoch {epoch+1}: {loss.item():.5f}\n')

### Vysvetlenie

Takto implementovaná trénovacia slučka je v súlade so zaužívanou štruktúrou trénovania sietí, prezentovanou aj na ostatných cvičeniach.

***

# Úloha č.8:

### Znenie

Implementujte funkciu, ktorá na vstupe dostáva:
1. **seed text** - vstupný text (začiatočné slová generovaného vtipu)
2. **words_to_generate** - počet slov na dogenerovanie
3. **sequence_len** - dľžka vstupnej sequencie do siete
4. **model** - váš predtrénovaný model

a vykonáva generovanie nového vtipu pomocou natrénovaného modelu.

Princíp fungovania funkcie je nasledovný:
1. Spracovať vstupný text na pole indexov
2. Iteratívne pre počet nových slov: <br/>
    2.1 Z posledných *sequence_len* hodnôt v poli predikovať nasledujúce slovo <br/>
    2.2 Predikované slovo pridať do poľa <br/>
3. Hodnoty vo výslednom poli previesť na ich textovú reprezentáciu

### Riešenie

In [ ]:
def generate_joke(seed_text, words_to_generate, model, sequence_len):
    clean_seed = clean_text(seed_text)
    seed_tokens = tokenizer(clean_seed)
    joke_indices = [token_to_index[token] for token in seed_tokens]
        
    for i in range(words_to_generate):
        input_sequence = joke_indices[-sequence_len:]
        x = torch.Tensor(input_sequence).reshape(-1, sequence_len).int().to(device)
         
        predicted = model(x)
        predicted_label = torch.argmax(predicted)
        
        joke_indices.append(predicted_label.item())

    text_words = [index_to_token[idx] for idx in joke_indices]
    return " ".join(text_words)

joke = generate_joke("Knock knock who's there?", 10, model, seq_len)
print(joke)

### Vysvetlenie

Uvedený kód implementuje funkciu tak, ako je opísaná v znení úlohy. Funkcia najprv vstupný text vyčistí, spracuje na tokeny a tie následne prevedie na pole ich indexov. Nové slová v požadovanom počte sú generované v cykle, pričom ako vstup do siete je použitých posledných *sequence_len* tokenov. T.j. ak chceme dogenerovať nasledujúce slová pre *seed_text* dlhší ako nami nastavená dĺžka sekvencie, nasledujúce slovo bude generované len na základe posledných  *sequence_len* tokenov.

Novo vygenerované slovo je následne pridané na koniec poľa tokenov textu a proces sa cyklicky opakuje, pokým nebol vygenerovaný požadovaný počet nových slov. Následne je celé pole *joke_indices* prekonvertované na text, za pomoci vytvoreného slovníka *index_to_token*. Spojený zoznam slov tvorí výstup funkcie.